# Ensemble Member Comparison: 01 vs 06

Compare tropical nights and hot heat days projections between CHESS-SCAPE ensemble members 01 and 06.

**Focus:** RCP6.0 and RCP8.5 bias-corrected, 2070s decade

In [44]:
import xarray as xr
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
import os

## Load Data

Load tropical nights and hot heat days for both ensemble members (01 and 06).

In [45]:
# Base path for chess-scape data
data_dir_85 = '../store/chess-scape/data/rcp85_bias-corrected'
data_dir_60 = '../store/chess-scape/data/rcp60_bias-corrected'

# File patterns
def get_file_path(ensemble, variable, rcp='85'):
    data_dir = data_dir_85 if rcp == '85' else data_dir_60
    return f"{data_dir}/{ensemble}/annual/chess-scape_rcp{rcp}_bias-corrected_{ensemble}_{variable}_uk_1km_annual_19801201-20801130.nc"

# Load RCP8.5 data
print("Loading RCP8.5 data...")
tropical_nights_01_85 = xr.open_dataset(get_file_path('01', 'tropical_nights', '85'))
tropical_nights_06_85 = xr.open_dataset(get_file_path('06', 'tropical_nights', '85'))
hot_heat_days_01_85 = xr.open_dataset(get_file_path('01', 'hot_heat_days', '85'))
hot_heat_days_06_85 = xr.open_dataset(get_file_path('06', 'hot_heat_days', '85'))

# Load RCP6.0 data
print("Loading RCP6.0 data...")
tropical_nights_01_60 = xr.open_dataset(get_file_path('01', 'tropical_nights', '60'))
tropical_nights_06_60 = xr.open_dataset(get_file_path('06', 'tropical_nights', '60'))
hot_heat_days_01_60 = xr.open_dataset(get_file_path('01', 'hot_heat_days', '60'))
hot_heat_days_06_60 = xr.open_dataset(get_file_path('06', 'hot_heat_days', '60'))

print("\n" + "="*60)
print("RCP8.5 - Tropical Nights (Ensemble 01):")
print(f"  Dimensions: {dict(tropical_nights_01_85.dims)}")
print(f"  Variables: {list(tropical_nights_01_85.data_vars)}")
print(f"  Decades: {tropical_nights_01_85.decade.values if 'decade' in tropical_nights_01_85.dims else 'N/A'}")

print("\nRCP6.0 - Tropical Nights (Ensemble 01):")
print(f"  Dimensions: {dict(tropical_nights_01_60.dims)}")
print(f"  Variables: {list(tropical_nights_01_60.data_vars)}")
print(f"  Decades: {tropical_nights_01_60.decade.values if 'decade' in tropical_nights_01_60.dims else 'N/A'}")
print("="*60)

Loading RCP8.5 data...
Loading RCP6.0 data...

RCP8.5 - Tropical Nights (Ensemble 01):
  Dimensions: {'decade': 2, 'y': 1057, 'x': 656}
  Variables: ['variable']
  Decades: [0 9]

RCP6.0 - Tropical Nights (Ensemble 01):
  Dimensions: {'decade': 2, 'y': 1057, 'x': 656}
  Variables: ['variable']
  Decades: [0 9]


/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/1824155528.py:26: FutureWarning:

The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.

/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/1824155528.py:31: FutureWarning:

The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.



## Visualisation Functions

In [46]:
def get_2070s_data(ds, var_name='variable'):
    """
    Extract the 2070s decade data (last timestep, index -1 or decade==9)
    """
    data = ds[var_name]
    if 'decade' in ds.dims:
        # Find index for 2070s (decade value 9 or last index)
        decades = ds.decade.values
        if 9 in decades:
            idx = np.where(decades == 9)[0][0]
        else:
            idx = -1  # Use last available decade
        return data.isel(decade=idx)
    elif 'time' in ds.dims:
        return data.isel(time=-1)
    return data


def plot_ensemble_difference(ds_01, ds_06, var_name='variable', title='Ensemble Difference (06 - 01)'):
    """
    Plot the difference between ensemble 06 and ensemble 01 for the 2070s decade.
    Positive values = ensemble 06 projects MORE of the variable.
    """
    # Get 2070s data
    data_01 = get_2070s_data(ds_01, var_name)
    data_06 = get_2070s_data(ds_06, var_name)
    
    # Calculate difference
    diff = data_06.values - data_01.values
    
    x = ds_01.x.values
    y = ds_01.y.values
    
    # Symmetric color scale
    finite = np.isfinite(diff)
    if finite.any():
        maxabs = np.nanmax(np.abs(diff[finite]))
        zmin, zmax = -maxabs, maxabs
    else:
        zmin, zmax = -1, 1
    
    # Diverging colorscale (blue = 01 higher, red = 06 higher)
    colorscale = [
        [0.0, '#2166ac'],   # Blue - ensemble 01 higher
        [0.45, '#92c5de'],
        [0.5, '#ffffff'],   # White - no difference
        [0.55, '#f4a582'],
        [1.0, '#b2182b']    # Red - ensemble 06 higher
    ]
    
    fig = go.Figure(go.Heatmap(
        z=diff,
        x=x,
        y=y,
        colorscale=colorscale,
        zmin=zmin,
        zmax=zmax,
        colorbar=dict(title='Difference<br>(days/year)'),
        hovertemplate="X: %{x:.0f}<br>Y: %{y:.0f}<br>Diff: %{z:.2f} days<extra></extra>"
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title="Easting (m)",
        yaxis_title="Northing (m)",
        width=800,
        height=700
    )
    
    # Print summary statistics
    valid = np.isfinite(diff)
    if valid.any():
        mean_diff = np.nanmean(diff[valid])
        median_diff = np.nanmedian(diff[valid])
        pct_06_higher = 100.0 * np.sum(diff[valid] > 0) / np.sum(valid)
        pct_01_higher = 100.0 * np.sum(diff[valid] < 0) / np.sum(valid)
        max_diff = np.nanmax(diff[valid])
        min_diff = np.nanmin(diff[valid])
        
        print(f"Summary Statistics:")
        print(f"  Mean difference: {mean_diff:+.3f} days/year")
        print(f"  Median difference: {median_diff:+.3f} days/year")
        print(f"  Range: [{min_diff:.2f}, {max_diff:.2f}] days/year")
        print(f"  Grid cells where Ensemble 06 > 01: {pct_06_higher:.1f}%")
        print(f"  Grid cells where Ensemble 01 > 06: {pct_01_higher:.1f}%")
    
    return fig


def plot_side_by_side(ds_01, ds_06, var_name='variable', title='Ensemble Comparison'):
    """
    Plot ensemble 01 and 06 side by side with synchronized color scales.
    """
    data_01 = get_2070s_data(ds_01, var_name).values
    data_06 = get_2070s_data(ds_06, var_name).values
    
    x = ds_01.x.values
    y = ds_01.y.values
    
    # Shared color scale based on both datasets
    all_data = np.concatenate([data_01.flatten(), data_06.flatten()])
    finite = np.isfinite(all_data)
    if finite.any():
        vmin = np.nanmin(all_data[finite])
        vmax = np.nanmax(all_data[finite])
    else:
        vmin, vmax = 0, 1
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Ensemble 01', 'Ensemble 06'),
        horizontal_spacing=0.1
    )
    
    # Ensemble 01
    fig.add_trace(
        go.Heatmap(
            z=data_01,
            x=x,
            y=y,
            colorscale='YlOrRd',
            zmin=vmin,
            zmax=vmax,
            showscale=False,
            hovertemplate="X: %{x:.0f}<br>Y: %{y:.0f}<br>Value: %{z:.2f}<extra></extra>"
        ),
        row=1, col=1
    )
    
    # Ensemble 06
    fig.add_trace(
        go.Heatmap(
            z=data_06,
            x=x,
            y=y,
            colorscale='YlOrRd',
            zmin=vmin,
            zmax=vmax,
            colorbar=dict(title='Days/year', x=1.02),
            hovertemplate="X: %{x:.0f}<br>Y: %{y:.0f}<br>Value: %{z:.2f}<extra></extra>"
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title=title,
        width=1200,
        height=600
    )
    
    # Print comparison stats
    valid_01 = np.isfinite(data_01)
    valid_06 = np.isfinite(data_06)
    
    print(f"Ensemble 01: mean={np.nanmean(data_01[valid_01]):.2f}, max={np.nanmax(data_01[valid_01]):.2f} days/year")
    print(f"Ensemble 06: mean={np.nanmean(data_06[valid_06]):.2f}, max={np.nanmax(data_06[valid_06]):.2f} days/year")
    
    return fig


def plot_value_vs_difference(ds_01, ds_06, var_name='variable', metric_name='Metric', min_threshold=0.5, use_percentage=True):
    """
    Scatter plot showing relationship between baseline value and ensemble difference (absolute or percentage).
    Helps identify if areas with higher values show bigger relative differences.
    
    Parameters:
    - min_threshold: minimum mean value (days/year) to include; filters out near-zero baselines (only for percentage mode)
    - use_percentage: if True, plot percentage difference; if False, plot absolute difference
    """
    data_01 = get_2070s_data(ds_01, var_name).values.flatten()
    data_06 = get_2070s_data(ds_06, var_name).values.flatten()
    
    # Use mean of both ensembles as the baseline value
    mean_value = (data_01 + data_06) / 2
    
    # Calculate difference
    abs_diff = data_06 - data_01
    
    if use_percentage:
        # Calculate percentage difference: (06 - 01) / mean * 100
        pct_diff = (abs_diff / mean_value) * 100.0
        
        # Filter valid points and exclude near-zero baselines
        valid = np.isfinite(mean_value) & np.isfinite(pct_diff) & (mean_value >= min_threshold)
        mean_value = mean_value[valid]
        diff_values = pct_diff[valid]
        
        print(f"Filtered out {np.sum(~valid)} grid cells with mean < {min_threshold} days/year")
        print(f"Analyzing {len(mean_value)} grid cells")
        
        y_label = '% Difference (06 - 01)'
        hover_label = '% Difference'
        hover_format = ':.1f'
        units = '%'
        diff_type = '% difference'
    else:
        # Use absolute difference
        valid = np.isfinite(mean_value) & np.isfinite(abs_diff)
        mean_value = mean_value[valid]
        diff_values = abs_diff[valid]
        
        print(f"Analyzing {len(mean_value)} grid cells")
        
        y_label = 'Absolute Difference (06 - 01) (days/year)'
        hover_label = 'Difference'
        hover_format = ':.2f'
        units = ' days'
        diff_type = 'absolute difference'
    
    # Subsample for performance if too many points
    if len(mean_value) > 10000:
        idx = np.random.choice(len(mean_value), 10000, replace=False)
        mean_value_sample = mean_value[idx]
        diff_sample = diff_values[idx]
    else:
        mean_value_sample = mean_value
        diff_sample = diff_values
    
    fig = go.Figure()
    
    # Scatter plot with density coloring
    fig.add_trace(go.Scattergl(
        x=mean_value_sample,
        y=diff_sample,
        mode='markers',
        marker=dict(
            size=4,
            color=mean_value_sample,
            colorscale='Viridis',
            opacity=0.5,
            colorbar=dict(title='Mean Value<br>(days/year)', x=1.02)
        ),
        hovertemplate=f"Mean Value: %{{x:.2f}} days<br>{hover_label}: %{{y{hover_format}}}{units}<extra></extra>"
    ))
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="No difference")
    
    # Add trend line
    z = np.polyfit(mean_value, diff_values, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(mean_value.min(), mean_value.max(), 100)
    
    slope_units = '%/day' if use_percentage else 'days/day'
    fig.add_trace(go.Scatter(
        x=x_trend,
        y=p(x_trend),
        mode='lines',
        line=dict(color='black', width=2, dash='dot'),
        name=f'Trend (slope={z[0]:.3f} {slope_units})'
    ))
    
    # Calculate correlation
    corr = np.corrcoef(mean_value, diff_values)[0, 1]
    
    title_type = '% Difference' if use_percentage else 'Absolute Difference'
    fig.update_layout(
        title=f'{metric_name}: Mean Value vs Ensemble {title_type}<br><sub>Correlation: r = {corr:.3f}</sub>',
        xaxis_title=f'Mean Value (days/year)',
        yaxis_title=y_label,
        width=800,
        height=500,
        showlegend=True
    )
    
    print(f"\nCorrelation between mean value and {diff_type}: r = {corr:.3f}")
    if abs(corr) > 0.5:
        if use_percentage:
            print(f"  → Strong {'positive' if corr > 0 else 'negative'} correlation: relative uncertainty {'increases' if corr > 0 else 'decreases'} with baseline value")
        else:
            print(f"  → Strong {'positive' if corr > 0 else 'negative'} correlation: absolute uncertainty {'increases' if corr > 0 else 'decreases'} with baseline value")
    elif abs(corr) > 0.3:
        print(f"  → Moderate {'positive' if corr > 0 else 'negative'} correlation")
    else:
        print(f"  → Weak correlation: ensemble uncertainty is fairly uniform across value ranges")
    
    # Print difference statistics
    spread = np.abs(diff_values)
    if use_percentage:
        print(f"\n% Difference Statistics:")
        print(f"  Mean absolute % difference: {np.mean(spread):.1f}%")
        print(f"  Median absolute % difference: {np.median(spread):.1f}%")
        print(f"  95th percentile: {np.percentile(spread, 95):.1f}%")
    else:
        print(f"\nAbsolute Difference Statistics:")
        print(f"  Mean absolute difference: {np.mean(spread):.2f} days/year")
        print(f"  Median absolute difference: {np.median(spread):.2f} days/year")
        print(f"  95th percentile: {np.percentile(spread, 95):.2f} days/year")
    
    return fig


def plot_ensemble_scatter(ds_01, ds_06, var_name='variable', metric_name='Metric'):
    """
    Scatter plot of ensemble 01 vs ensemble 06 values with 1:1 line.
    Points above the line = ensemble 06 higher, below = ensemble 01 higher.
    """
    data_01 = get_2070s_data(ds_01, var_name).values.flatten()
    data_06 = get_2070s_data(ds_06, var_name).values.flatten()
    
    # Filter valid points
    valid = np.isfinite(data_01) & np.isfinite(data_06)
    data_01 = data_01[valid]
    data_06 = data_06[valid]
    
    # Subsample for performance
    if len(data_01) > 10000:
        idx = np.random.choice(len(data_01), 10000, replace=False)
        data_01_sample = data_01[idx]
        data_06_sample = data_06[idx]
    else:
        data_01_sample = data_01
        data_06_sample = data_06
    
    fig = go.Figure()
    
    # Scatter plot
    fig.add_trace(go.Scattergl(
        x=data_01_sample,
        y=data_06_sample,
        mode='markers',
        marker=dict(
            size=4,
            color='#1f77b4',
            opacity=0.4
        ),
        name='Grid cells',
        hovertemplate="Ensemble 01: %{x:.2f}<br>Ensemble 06: %{y:.2f}<extra></extra>"
    ))
    
    # 1:1 line
    max_val = max(data_01.max(), data_06.max())
    min_val = min(data_01.min(), data_06.min())
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(color='red', width=2, dash='dash'),
        name='1:1 line (perfect agreement)'
    ))
    
    fig.update_layout(
        title=f'{metric_name}: Ensemble 01 vs Ensemble 06',
        xaxis_title='Ensemble 01 (days/year)',
        yaxis_title='Ensemble 06 (days/year)',
        width=700,
        height=600,
        showlegend=True
    )
    
    # Calculate agreement stats
    pct_above = 100.0 * np.sum(data_06 > data_01) / len(data_01)
    pct_below = 100.0 * np.sum(data_06 < data_01) / len(data_01)
    
    print(f"Agreement Statistics:")
    print(f"  {pct_above:.1f}% of grid cells: Ensemble 06 > Ensemble 01 (above 1:1 line)")
    print(f"  {pct_below:.1f}% of grid cells: Ensemble 01 > Ensemble 06 (below 1:1 line)")
    
    return fig


def create_comparison_dashboard(ds_01, ds_06, var_name='variable', metric_name='Metric'):
    """
    Create an interactive dashboard with tabs for different views.
    """
    # Pre-compute all figures
    fig_diff = plot_ensemble_difference(ds_01, ds_06, var_name, 
                                        title=f'{metric_name}: Ensemble Difference (06 - 01) - 2070s')
    fig_side = plot_side_by_side(ds_01, ds_06, var_name,
                                 title=f'{metric_name}: Side-by-Side Comparison - 2070s')
    
    # Create tab widget
    tab = widgets.Tab()
    
    out_diff = widgets.Output()
    out_side = widgets.Output()
    
    with out_diff:
        fig_diff.show()
    
    with out_side:
        fig_side.show()
    
    tab.children = [out_diff, out_side]
    tab.set_title(0, 'Difference Map')
    tab.set_title(1, 'Side-by-Side')
    
    return tab

## RCP8.5: Tropical Nights Comparison

Tropical nights = nights where minimum temperature ≥ 20°C

In [47]:
print("=" * 60)
print("RCP8.5 - TROPICAL NIGHTS - 2070s Comparison")
print("=" * 60)

tropical_dashboard_85 = create_comparison_dashboard(
    tropical_nights_01_85, 
    tropical_nights_06_85, 
    var_name='variable',
    metric_name='Tropical Nights (RCP8.5)'
)
display(tropical_dashboard_85)

RCP8.5 - TROPICAL NIGHTS - 2070s Comparison
Summary Statistics:
  Mean difference: -0.455 days/year
  Median difference: +0.000 days/year
  Range: [-7.70, 10.90] days/year
  Grid cells where Ensemble 06 > 01: 2.6%
  Grid cells where Ensemble 01 > 06: 27.3%
Ensemble 01: mean=1.56, max=48.10 days/year
Ensemble 06: mean=1.11, max=53.60 days/year


## RCP6.0: Tropical Nights Comparison

Tropical nights = nights where minimum temperature ≥ 20°C

In [48]:
print("=" * 60)
print("RCP6.0 - TROPICAL NIGHTS - 2070s Comparison")
print("=" * 60)

tropical_dashboard_60 = create_comparison_dashboard(
    tropical_nights_01_60, 
    tropical_nights_06_60, 
    var_name='variable',
    metric_name='Tropical Nights (RCP6.0)'
)
display(tropical_dashboard_60)

RCP6.0 - TROPICAL NIGHTS - 2070s Comparison
Summary Statistics:
  Mean difference: -0.209 days/year
  Median difference: +0.000 days/year
  Range: [-10.60, 1.40] days/year
  Grid cells where Ensemble 06 > 01: 2.5%
  Grid cells where Ensemble 01 > 06: 20.0%
Ensemble 01: mean=0.36, max=30.20 days/year
Ensemble 06: mean=0.15, max=21.20 days/year


## Distribution Analysis: Tropical Nights

Comparing ensemble scatter and value vs difference patterns across both RCP scenarios.

In [49]:
# Scatter plot: Ensemble 01 vs Ensemble 06 for both RCP scenarios
print("RCP8.5:")
fig_scatter_85 = plot_ensemble_scatter(tropical_nights_01_85, tropical_nights_06_85, 
                                        var_name='variable', metric_name='Tropical Nights (RCP8.5)')
fig_scatter_85.show()

print("\nRCP6.0:")
fig_scatter_60 = plot_ensemble_scatter(tropical_nights_01_60, tropical_nights_06_60, 
                                        var_name='variable', metric_name='Tropical Nights (RCP6.0)')
fig_scatter_60.show()

RCP8.5:
Agreement Statistics:
  2.6% of grid cells: Ensemble 06 > Ensemble 01 (above 1:1 line)
  27.3% of grid cells: Ensemble 01 > Ensemble 06 (below 1:1 line)



RCP6.0:
Agreement Statistics:
  2.5% of grid cells: Ensemble 06 > Ensemble 01 (above 1:1 line)
  20.0% of grid cells: Ensemble 01 > Ensemble 06 (below 1:1 line)


In [50]:
print("RCP8.5 - Percentage Difference:")
fig_val_diff_pct_85 = plot_value_vs_difference(tropical_nights_01_85, tropical_nights_06_85,
                                                var_name='variable', metric_name='Tropical Nights (RCP8.5)',
                                                use_percentage=True)
fig_val_diff_pct_85.show()

print("RCP8.5 - Absolute Difference:")
fig_val_diff_abs_85 = plot_value_vs_difference(tropical_nights_01_85, tropical_nights_06_85,
                                                var_name='variable', metric_name='Tropical Nights (RCP8.5)',
                                                use_percentage=False)
fig_val_diff_abs_85.show()

RCP8.5 - Percentage Difference:
Filtered out 521302 grid cells with mean < 0.5 days/year
Analyzing 172090 grid cells

Correlation between mean value and % difference: r = 0.284
  → Weak correlation: ensemble uncertainty is fairly uniform across value ranges

% Difference Statistics:
  Mean absolute % difference: 50.5%
  Median absolute % difference: 40.0%
  95th percentile: 140.0%


/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/4286139616.py:181: RuntimeWarning:

invalid value encountered in divide



RCP8.5 - Absolute Difference:
Analyzing 693392 grid cells

Correlation between mean value and absolute difference: r = -0.622
  → Strong negative correlation: absolute uncertainty decreases with baseline value

Absolute Difference Statistics:
  Mean absolute difference: 0.48 days/year
  Median absolute difference: 0.00 days/year
  95th percentile: 3.10 days/year


In [51]:
print("\nRCP6.0 - Percentage Difference:")
fig_val_diff_pct_60 = plot_value_vs_difference(tropical_nights_01_60, tropical_nights_06_60,
                                                var_name='variable', metric_name='Tropical Nights (RCP6.0)',
                                                use_percentage=True)
fig_val_diff_pct_60.show()

print("\nRCP6.0 - Absolute Difference:")
fig_val_diff_abs_60 = plot_value_vs_difference(tropical_nights_01_60, tropical_nights_06_60,
                                                var_name='variable', metric_name='Tropical Nights (RCP6.0)',
                                                use_percentage=False)
fig_val_diff_abs_60.show()


RCP6.0 - Percentage Difference:
Filtered out 597045 grid cells with mean < 0.5 days/year
Analyzing 96347 grid cells

Correlation between mean value and % difference: r = 0.163
  → Weak correlation: ensemble uncertainty is fairly uniform across value ranges

% Difference Statistics:
  Mean absolute % difference: 96.3%
  Median absolute % difference: 102.7%
  95th percentile: 161.9%


/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/4286139616.py:181: RuntimeWarning:

invalid value encountered in divide




RCP6.0 - Absolute Difference:
Analyzing 693392 grid cells

Correlation between mean value and absolute difference: r = -0.874
  → Strong negative correlation: absolute uncertainty decreases with baseline value

Absolute Difference Statistics:
  Mean absolute difference: 0.22 days/year
  Median absolute difference: 0.00 days/year
  95th percentile: 1.60 days/year


## RCP8.5: Hot Heat Days Comparison

Hot heat days = days where maximum temperature ≥ 30°C

In [52]:
print("=" * 60)
print("RCP8.5 - HOT HEAT DAYS - 2070s Comparison")
print("=" * 60)

hot_days_dashboard_85 = create_comparison_dashboard(
    hot_heat_days_01_85, 
    hot_heat_days_06_85, 
    var_name='variable',
    metric_name='Hot Heat Days (RCP8.5)'
)
display(hot_days_dashboard_85)

RCP8.5 - HOT HEAT DAYS - 2070s Comparison
Summary Statistics:
  Mean difference: +1.237 days/year
  Median difference: +0.000 days/year
  Range: [-2.34, 15.46] days/year
  Grid cells where Ensemble 06 > 01: 26.1%
  Grid cells where Ensemble 01 > 06: 5.7%
Ensemble 01: mean=3.38, max=31.46 days/year
Ensemble 06: mean=4.62, max=44.10 days/year


## RCP6.0: Hot Heat Days Comparison

Hot heat days = days where maximum temperature ≥ 30°C

In [53]:
print("=" * 60)
print("RCP6.0 - HOT HEAT DAYS - 2070s Comparison")
print("=" * 60)

hot_days_dashboard_60 = create_comparison_dashboard(
    hot_heat_days_01_60, 
    hot_heat_days_06_60, 
    var_name='variable',
    metric_name='Hot Heat Days (RCP6.0)'
)
display(hot_days_dashboard_60)

RCP6.0 - HOT HEAT DAYS - 2070s Comparison
Summary Statistics:
  Mean difference: +0.079 days/year
  Median difference: +0.000 days/year
  Range: [-2.60, 4.70] days/year
  Grid cells where Ensemble 06 > 01: 15.4%
  Grid cells where Ensemble 01 > 06: 10.1%
Ensemble 01: mean=0.91, max=12.20 days/year
Ensemble 06: mean=0.99, max=14.00 days/year


## Distribution Analysis: Hot Heat Days

Comparing ensemble scatter and value vs difference patterns across both RCP scenarios.

In [54]:
# Scatter plot: Ensemble 01 vs Ensemble 06 for both RCP scenarios
print("RCP8.5:")
fig_scatter_hot_85 = plot_ensemble_scatter(hot_heat_days_01_85, hot_heat_days_06_85, 
                                            var_name='variable', metric_name='Hot Heat Days (RCP8.5)')
fig_scatter_hot_85.show()

print("\nRCP6.0:")
fig_scatter_hot_60 = plot_ensemble_scatter(hot_heat_days_01_60, hot_heat_days_06_60, 
                                            var_name='variable', metric_name='Hot Heat Days (RCP6.0)')
fig_scatter_hot_60.show()

RCP8.5:
Agreement Statistics:
  26.1% of grid cells: Ensemble 06 > Ensemble 01 (above 1:1 line)
  5.7% of grid cells: Ensemble 01 > Ensemble 06 (below 1:1 line)



RCP6.0:
Agreement Statistics:
  15.4% of grid cells: Ensemble 06 > Ensemble 01 (above 1:1 line)
  10.1% of grid cells: Ensemble 01 > Ensemble 06 (below 1:1 line)


In [55]:
print("RCP8.5 - Percentage Difference:")
fig_val_diff_hot_pct_85 = plot_value_vs_difference(hot_heat_days_01_85, hot_heat_days_06_85,
                                                    var_name='variable', metric_name='Hot Heat Days (RCP8.5)',
                                                    use_percentage=True)
fig_val_diff_hot_pct_85.show()

print("RCP8.5 - Absolute Difference:")
fig_val_diff_hot_abs_85 = plot_value_vs_difference(hot_heat_days_01_85, hot_heat_days_06_85,
                                                    var_name='variable', metric_name='Hot Heat Days (RCP8.5)',
                                                    use_percentage=False)
fig_val_diff_hot_abs_85.show()

RCP8.5 - Percentage Difference:
Filtered out 493308 grid cells with mean < 0.5 days/year
Analyzing 200084 grid cells

Correlation between mean value and % difference: r = 0.235
  → Weak correlation: ensemble uncertainty is fairly uniform across value ranges

% Difference Statistics:
  Mean absolute % difference: 32.0%
  Median absolute % difference: 30.9%
  95th percentile: 61.7%


/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/4286139616.py:181: RuntimeWarning:

invalid value encountered in divide



RCP8.5 - Absolute Difference:
Analyzing 693392 grid cells

Correlation between mean value and absolute difference: r = 0.974
  → Strong positive correlation: absolute uncertainty increases with baseline value

Absolute Difference Statistics:
  Mean absolute difference: 1.28 days/year
  Median absolute difference: 0.00 days/year
  95th percentile: 8.56 days/year


In [56]:
print("\nRCP6.0 - Percentage Difference:")
fig_val_diff_hot_pct_60 = plot_value_vs_difference(hot_heat_days_01_60, hot_heat_days_06_60,
                                                    var_name='variable', metric_name='Hot Heat Days (RCP6.0)',
                                                    use_percentage=True)
fig_val_diff_hot_pct_60.show()

print("\nRCP6.0 - Absolute Difference:")
fig_val_diff_hot_abs_60 = plot_value_vs_difference(hot_heat_days_01_60, hot_heat_days_06_60,
                                                    var_name='variable', metric_name='Hot Heat Days (RCP6.0)',
                                                    use_percentage=False)
fig_val_diff_hot_abs_60.show()




RCP6.0 - Percentage Difference:
Filtered out 546232 grid cells with mean < 0.5 days/year
Analyzing 147160 grid cells

Correlation between mean value and % difference: r = 0.041
  → Weak correlation: ensemble uncertainty is fairly uniform across value ranges

% Difference Statistics:
  Mean absolute % difference: 21.9%
  Median absolute % difference: 14.5%
  95th percentile: 66.7%


/var/folders/xl/g4455wnj0w5f31hprr7rvdwm0000gp/T/ipykernel_58296/4286139616.py:181: RuntimeWarning:

invalid value encountered in divide




RCP6.0 - Absolute Difference:
Analyzing 693392 grid cells

Correlation between mean value and absolute difference: r = 0.429
  → Moderate positive correlation

Absolute Difference Statistics:
  Mean absolute difference: 0.15 days/year
  Median absolute difference: 0.00 days/year
  95th percentile: 0.90 days/year


## Summary Statistics Table

In [57]:
def compute_summary_stats(ds_01, ds_06, var_name='variable'):
    """Compute summary statistics for ensemble comparison"""
    data_01 = get_2070s_data(ds_01, var_name).values
    data_06 = get_2070s_data(ds_06, var_name).values
    diff = data_06 - data_01
    
    valid_01 = np.isfinite(data_01)
    valid_06 = np.isfinite(data_06)
    valid_diff = np.isfinite(diff)
    
    return {
        'Ensemble 01 Mean': np.nanmean(data_01[valid_01]),
        'Ensemble 06 Mean': np.nanmean(data_06[valid_06]),
        'Ensemble 01 Max': np.nanmax(data_01[valid_01]),
        'Ensemble 06 Max': np.nanmax(data_06[valid_06]),
        'Mean Difference': np.nanmean(diff[valid_diff]),
        'Max Abs Difference': np.nanmax(np.abs(diff[valid_diff])),
        '% Cells 06 > 01': 100.0 * np.sum(diff[valid_diff] > 0) / np.sum(valid_diff),
    }

# Compute stats for RCP8.5
tropical_stats_85 = compute_summary_stats(tropical_nights_01_85, tropical_nights_06_85)
hot_days_stats_85 = compute_summary_stats(hot_heat_days_01_85, hot_heat_days_06_85)

# Compute stats for RCP6.0
tropical_stats_60 = compute_summary_stats(tropical_nights_01_60, tropical_nights_06_60)
hot_days_stats_60 = compute_summary_stats(hot_heat_days_01_60, hot_heat_days_06_60)

# Display as HTML tables
html = """
<style>
    .stats-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; margin-bottom: 20px; }
    .stats-table th, .stats-table td { border: 1px solid #ddd; padding: 12px; text-align: right; }
    .stats-table th { background-color: #4472C4; color: white; }
    .stats-table tr:nth-child(even) { background-color: #403e3e; }
    .stats-table td:first-child { text-align: left; font-weight: bold; }
</style>
<h3>2070s Ensemble Comparison Summary (Bias-Corrected)</h3>

<h4>RCP8.5</h4>
<table class="stats-table">
    <tr><th>Metric</th><th>Tropical Nights</th><th>Hot Heat Days</th></tr>
"""

for key in tropical_stats_85.keys():
    html += f"<tr><td>{key}</td><td>{tropical_stats_85[key]:.2f}</td><td>{hot_days_stats_85[key]:.2f}</td></tr>"

html += """</table>

<h4>RCP6.0</h4>
<table class="stats-table">
    <tr><th>Metric</th><th>Tropical Nights</th><th>Hot Heat Days</th></tr>
"""

for key in tropical_stats_60.keys():
    html += f"<tr><td>{key}</td><td>{tropical_stats_60[key]:.2f}</td><td>{hot_days_stats_60[key]:.2f}</td></tr>"

html += "</table>"

display(HTML(html))

Metric,Tropical Nights,Hot Heat Days
Ensemble 01 Mean,1.56,3.38
Ensemble 06 Mean,1.11,4.62
Ensemble 01 Max,48.10,31.46
Ensemble 06 Max,53.60,44.10
Mean Difference,-0.45,1.24
Max Abs Difference,10.90,15.46
% Cells 06 > 01,2.61,26.12
Metric,Tropical Nights,Hot Heat Days
Ensemble 01 Mean,0.36,0.91
Ensemble 06 Mean,0.15,0.99
